In [1]:
import os

import pandas as pd
import psycopg
from dotenv import load_dotenv

load_dotenv()

conn = psycopg.connect(
    host=os.getenv("POSTGRES_HOST", "localhost"),
    port=os.getenv("POSTGRES_PORT", "5436"),
    dbname=os.getenv("POSTGRES_DB"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
)

# a bronze tem um snapshot do resumo por execução do extrator; pegamos o mais recente
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT id, query_params, payload, ingested_at
        FROM bronze.sentry_raw
        WHERE source = 'sentry_resumo'
        ORDER BY id DESC
        LIMIT 1
        """
    )
    linha_id, params, payload, ingested_at = cur.fetchone()

print("linha:", linha_id, "| gravada em:", ingested_at)
print("parâmetros:", params)
print("assinatura:", payload["signature"])
print("count:", payload["count"], "| registros em data:", len(payload["data"]))

linha: 15 | gravada em: 2026-11-17 03:59:29.987936+00:00
parâmetros: {}
assinatura: {'source': 'NASA/JPL Sentry Data API', 'version': '2.0'}
count: 2251 | registros em data: 2251


In [2]:
# aqui cada registro já é um dicionário, então o pandas descobre as colunas sozinho
resumo_df = pd.DataFrame(payload["data"])

print(resumo_df.shape)
resumo_df.head()

(2251, 14)


,h,id,ip,des,n_imp,range,v_inf,ps_cum,ps_max,ts_max,diameter,fullname,last_obs,last_obs_jd
0,26.32,bK11T00O,2.937e-06,2011 TO,1,2064-2064,8.55,-6.15,-6.15,0,0.018,(2011 TO),2013-01-12,2456304.5
1,26.51,bK12BA2A,1.22522146e-05,2012 BA102,20,2103-2122,7.21203952320587,-6.03,-6.27,0,0.017,(2012 BA102),2012-02-21,2455978.5
2,19.92,bK14HJ7N,6.5351e-09,2014 HN197,12,2069-2118,21.9988508209515,-5.50,-5.98,0,0.35,(2014 HN197),2014-04-27,2456774.5
3,24.86,bK09D00V,3.1018e-08,2009 DV,2,2044-2102,22.0982648784577,-7.24,-7.37,0,0.036,(2009 DV),2009-02-24,2454886.5
4,27.85,bK09E01H,5.902e-10,2009 EH1,1,2068-2068,10.69,-10.56,-10.56,0,0.0091,(2009 EH1),2009-03-05,2454895.5


In [3]:
resumo_df.dtypes

h                str
id               str
ip               str
des              str
n_imp          int64
range            str
v_inf            str
ps_cum           str
ps_max           str
ts_max           str
diameter         str
fullname         str
last_obs         str
last_obs_jd      str
dtype: object

In [7]:
resumo = resumo_df.copy()   # copia para não mexer no original

# colunas decimais: h, ip, v_inf, ps_cum, ps_max, diameter, last_obs_jd
# (o pandas lê notação científica como 1.2e-05 sem problema)
colunas_float = ["h", "ip", "v_inf", "ps_cum", "ps_max", "diameter", "last_obs_jd"]
resumo[colunas_float] = resumo[colunas_float].apply(lambda col: pd.to_numeric(col, errors="coerce"))

# ts_max é a escala de Torino (0 a 10), então é um número inteiro
resumo["ts_max"] = pd.to_numeric(resumo["ts_max"], errors="coerce").astype("Int64")

# última observação: de "2013-01-12" para data
resumo["last_obs"] = pd.to_datetime(resumo["last_obs"], format="%Y-%m-%d", errors="coerce")

# range vem como "2064-2118": separamos em dois anos
anos = resumo["range"].str.split("-", expand=True)
resumo["ano_min"] = anos[0].astype("int64")
resumo["ano_max"] = anos[1].astype("int64")

# espaços sobrando no nome completo, como no CAD
resumo["fullname"] = resumo["fullname"].str.strip()

# id continua texto: é um código de ordenação (ex.: "bK11T00O"), não um número
resumo.dtypes

h                     float64
id                        str
ip                    float64
des                       str
n_imp                   int64
range                     str
v_inf                 float64
ps_cum                float64
ps_max                float64
ts_max                  Int64
diameter              float64
fullname                  str
last_obs       datetime64[us]
last_obs_jd           float64
ano_min                 int64
ano_max                 int64
dtype: object

In [8]:
resumo.isna().sum()

h              0
id             0
ip             0
des            0
n_imp          0
range          0
v_inf          6
ps_cum         0
ps_max         0
ts_max         2
diameter       0
fullname       0
last_obs       1
last_obs_jd    0
ano_min        0
ano_max        0
dtype: int64

In [9]:
# linhas em que algum desses três campos ficou ausente depois da conversão
faltas = resumo[["v_inf", "ts_max", "last_obs"]].isna().any(axis=1)
print("linhas com falta:", faltas.sum())

# valores ORIGINAIS (resumo_df, antes da conversão) dessas linhas
colunas = ["des", "fullname", "h", "ip", "n_imp", "v_inf", "ts_max", "last_obs", "last_obs_jd"]
resumo_df.loc[faltas, colunas]

linhas com falta: 8


,des,fullname,h,ip,n_imp,v_inf,ts_max,last_obs,last_obs_jd
267,2006 RH120,(2006 RH120),29.64,0.00816650175,119,NaN,0,2007-06-22,2454273.5
1455,101955,101955 Bennu (1999 RQ36),20.63,0.000571699999999996,157,5.9916984432395,NaN,2020-10-3.80160,2459126.3016
1474,2022 NX1,(2022 NX1),28.03,0.00317246615,50,NaN,0,2022-11-21,2459904.5
1588,2026 QQ8,(2026 QQ8),31.37,0.008082093,26,NaN,0,2026-11-15,2461359.5
1606,2020 CD3,(2020 CD3),31.75,0.0252338159446,115,NaN,0,2020-05-20,2458989.5
1685,29075,29075 (1950 DA),17.94,3.401e-04,1,14.10,NaN,2026-11-05,2461349.5
2183,2010 VP139,(2010 VP139),29.29,1.738e-07,1,NaN,0,2010-11-14,2455514.5
2214,2026 TQ128,(2026 TQ128),31.70,0.0172242205,473,NaN,0,2026-10-16,2461329.5


In [10]:
# repr mostra a diferença: '' (texto vazio) aparece com aspas, None aparece sem
resumo_df.loc[faltas, ["v_inf", "ts_max", "last_obs"]].map(repr)

,v_inf,ts_max,last_obs
267,'NaN','0','2007-06-22'
1455,'5.9916984432395',nan,'2020-10-3.80160'
1474,'NaN','0','2022-11-21'
1588,'NaN','0','2026-11-15'
1606,'NaN','0','2020-05-20'
1685,'14.10',nan,'2026-11-05'
2183,'NaN','0','2010-11-14'
2214,'NaN','0','2026-10-16'


In [11]:
# next(...) pega o primeiro registro que satisfaz a condição
for des in ["101955", "29075"]:
    registro = next(r for r in payload["data"] if r["des"] == des)
    print(des, "->", registro, "\n")

101955 -> {'h': '20.63', 'id': 'a0101955', 'ip': '0.000571699999999996', 'des': '101955', 'n_imp': 157, 'range': '2178-2290', 'v_inf': '5.9916984432395', 'ps_cum': '-1.40', 'ps_max': '-1.58', 'ts_max': None, 'diameter': '0.49', 'fullname': '101955 Bennu (1999 RQ36)', 'last_obs': '2020-10-3.80160', 'last_obs_jd': '2459126.3016'} 

29075 -> {'h': '17.94', 'id': 'a0029075', 'ip': '3.401e-04', 'des': '29075', 'n_imp': 1, 'range': '2880-2880', 'v_inf': '14.10', 'ps_cum': '-0.98', 'ps_max': '-0.98', 'ts_max': None, 'diameter': '1.3', 'fullname': '29075 (1950 DA)', 'last_obs': '2026-11-05', 'last_obs_jd': '2461349.5'} 



In [12]:
# só a parte antes do ponto: "2020-10-3.80160" vira "2020-10-3"
# (datas sem ponto, como "2013-01-12", continuam iguais)
data_texto = resumo_df["last_obs"].str.split(".").str[0]

# format="mixed" lê cada valor separadamente, aceitando dia sem zero à esquerda
resumo["last_obs"] = pd.to_datetime(data_texto, format="mixed")

print("last_obs ausentes:", resumo["last_obs"].isna().sum())
resumo.loc[resumo["des"] == "101955", ["des", "last_obs", "last_obs_jd"]]

last_obs ausentes: 0


,des,last_obs,last_obs_jd
1455,101955,2020-10-03,2.459126e+06


In [13]:
for des in ["101955", "29075"]:
    registro = next(r for r in payload["data"] if r["des"] == des)
    print(des)
    print("  chaves:", sorted(registro.keys()))
    print("  tem ts_max:", "ts_max" in registro)

101955
  chaves: ['des', 'diameter', 'fullname', 'h', 'id', 'ip', 'last_obs', 'last_obs_jd', 'n_imp', 'ps_cum', 'ps_max', 'range', 'ts_max', 'v_inf']
  tem ts_max: True
29075
  chaves: ['des', 'diameter', 'fullname', 'h', 'id', 'ip', 'last_obs', 'last_obs_jd', 'n_imp', 'ps_cum', 'ps_max', 'range', 'ts_max', 'v_inf']
  tem ts_max: True


In [14]:
from collections import Counter

# quantos registros têm cada conjunto de chaves
formatos = Counter(tuple(sorted(r.keys())) for r in payload["data"])

for chaves, qtd in formatos.items():
    print(qtd, "registros com", len(chaves), "chaves")

2251 registros com 14 chaves


In [15]:
# repr mostra a diferença entre None, '' e 'NaN'
for des in ["101955", "29075", "2006 RH120"]:
    registro = next(r for r in payload["data"] if r["des"] == des)
    print(des, "| ts_max =", repr(registro["ts_max"]), "| v_inf =", repr(registro["v_inf"]))

101955 | ts_max = None | v_inf = '5.9916984432395'
29075 | ts_max = None | v_inf = '14.10'
2006 RH120 | ts_max = '0' | v_inf = 'NaN'


In [16]:
# para cada campo, quantos registros têm valor de cada tipo (str, int, NoneType...)
for campo in payload["data"][0].keys():
    contagem = Counter(type(r[campo]).__name__ for r in payload["data"])
    print(campo, dict(contagem))

# e quantos v_inf vieram com o texto "NaN"
print("v_inf == 'NaN':", sum(r["v_inf"] == "NaN" for r in payload["data"]))

h {'str': 2251}
id {'str': 2251}
ip {'str': 2251}
des {'str': 2251}
n_imp {'int': 2251}
range {'str': 2251}
v_inf {'str': 2251}
ps_cum {'str': 2251}
ps_max {'str': 2251}
ts_max {'str': 2249, 'NoneType': 2}
diameter {'str': 2251}
fullname {'str': 2251}
last_obs {'str': 2251}
last_obs_jd {'str': 2251}
v_inf == 'NaN': 6


In [17]:
# cada objeto deve aparecer uma vez no resumo
print("des duplicados:", resumo["des"].duplicated().sum())
print("id duplicados:", resumo["id"].duplicated().sum())

des duplicados: 0
id duplicados: 0


In [18]:
# probabilidade de impacto: maior que 0 e no máximo 1
print("ip fora de (0, 1]:", (~resumo["ip"].between(0, 1, inclusive="right")).sum())

# todo objeto da Sentry tem pelo menos 1 impacto potencial
print("n_imp < 1:", (resumo["n_imp"] < 1).sum())

# o primeiro ano dos impactos não pode ser maior que o último
print("ano_min > ano_max:", (resumo["ano_min"] > resumo["ano_max"]).sum())

# Palermo cumulativo deve ser >= ao máximo individual (somar probabilidades só aumenta)
# a tolerância 0.01 evita acusar diferenças de arredondamento
print("ps_cum < ps_max:", (resumo["ps_cum"] < resumo["ps_max"] - 0.01).sum())

# Torino vai de 0 a 10
print("ts_max fora de 0-10:", (~resumo["ts_max"].dropna().between(0, 10)).sum())

ip fora de (0, 1]: 0
n_imp < 1: 0
ano_min > ano_max: 0
ps_cum < ps_max: 0
ts_max fora de 0-10: 0


In [19]:
# a última observação não pode estar no futuro
print("última observação mais recente:", resumo["last_obs"].max())
print("anos dos impactos:", resumo["ano_min"].min(), "a", resumo["ano_max"].max())

resumo[["h", "diameter", "v_inf", "ip", "n_imp"]].describe()

última observação mais recente: 2026-11-16 00:00:00
anos dos impactos: 2026 a 2880


,h,diameter,v_inf,ip,n_imp
count,2251.000000,2251.000000,2245.000000,2.251000e+03,2251.000000
mean,27.285020,0.019368,10.998890,1.392191e-04,21.238561
std,1.871302,0.047793,5.608349,2.301966e-03,52.102783
min,17.630000,0.000990,0.410758,1.008000e-10,1.000000
25%,26.190000,0.006800,7.240000,2.050639e-07,2.000000
50%,27.370000,0.011000,9.951516,2.315870e-06,5.000000
75%,28.495000,0.020000,14.136852,2.094887e-05,19.000000
max,32.670000,1.300000,40.250000,1.026373e-01,972.000000


In [20]:
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT id, query_params, payload, ingested_at
        FROM bronze.sentry_raw
        WHERE source = 'sentry_vi'
        ORDER BY id DESC
        LIMIT 1
        """
    )
    linha_id, params_vi, payload_vi, ingested_at_vi = cur.fetchone()

print("linha:", linha_id, "| gravada em:", ingested_at_vi)
print("parâmetros:", params_vi)
print("assinatura:", payload_vi["signature"])
print("count:", payload_vi["count"], "| registros em data:", len(payload_vi["data"]))

linha: 16 | gravada em: 2026-11-17 03:59:49.328578+00:00
parâmetros: {'all': 'true'}
assinatura: {'source': 'NASA/JPL Sentry Data API', 'version': '2.0'}
count: 47808 | registros em data: 47808


In [21]:
# quantos registros têm cada conjunto de chaves
formatos_vi = Counter(tuple(sorted(r.keys())) for r in payload_vi["data"])

for chaves, qtd in formatos_vi.items():
    print(qtd, "registros:", chaves, "\n")

# e como o campo "method" se distribui
print(Counter(r.get("method") for r in payload_vi["data"]))

47651 registros: ('date', 'des', 'energy', 'fullname', 'id', 'ip', 'method', 'ps', 'sigma_vi', 'ts') 

157 registros: ('date', 'des', 'energy', 'fullname', 'id', 'ip', 'method', 'ps', 'sigma_mc', 'ts') 

Counter({'IOBS': 47651, 'MC': 157})


In [22]:
vi_df = pd.DataFrame(payload_vi["data"])   # chaves ausentes viram NaN

print(vi_df.shape)
display(vi_df.head())
vi_df.dtypes

(47808, 11)


,id,ip,ps,ts,des,date,energy,method,fullname,sigma_vi,sigma_mc
0,bK20K00U,9.452e-06,-7.08,0,2020 KU,2108-05-24.69,8.372e-03,IOBS,(2020 KU),1.3343,NaN
1,bK20P05X,1.820e-08,-10.26,0,2020 PX5,2118-08-13.56,2.509e-03,IOBS,(2020 PX5),2.6178,NaN
2,bK09SH1T,1.118e-06,-6.79,0,2009 ST171,2066-09-30.09,1.111e-01,IOBS,(2009 ST171),3.3702,NaN
3,bK01SS6D,6.222e-08,-7.34,0,2001 SD286,2080-09-23.22,1.255e+00,IOBS,(2001 SD286),2.2077,NaN
4,bK16S02U,3.030e-07,-8.07,0,2016 SU2,2120-09-26.19,4.175e-02,IOBS,(2016 SU2),1.3479,NaN


id          str
ip          str
ps          str
ts          str
des         str
date        str
energy      str
method      str
fullname    str
sigma_vi    str
sigma_mc    str
dtype: object

In [23]:
vi = vi_df.copy()   # copia para não mexer no original

# colunas numéricas (chegam como texto)
colunas_float = ["ip", "ps", "energy", "sigma_vi", "sigma_mc"]
vi[colunas_float] = vi[colunas_float].apply(pd.to_numeric)

# ts é inteiro; Int64 (com I maiúsculo) aceita nulo, caso apareça algum
vi["ts"] = pd.to_numeric(vi["ts"]).astype("Int64")

# date vem como "2108-05-24.69" (data + fração do dia): guardamos só a data
data_texto = vi["date"].str.split(".").str[0]
vi["data_impacto"] = pd.to_datetime(data_texto, format="%Y-%m-%d")

vi.dtypes

id                         str
ip                     float64
ps                     float64
ts                       Int64
des                        str
date                       str
energy                 float64
method                     str
fullname                   str
sigma_vi               float64
sigma_mc               float64
data_impacto    datetime64[us]
dtype: object

In [24]:
# compara nulos antes e depois: se "criados" > 0, a conversão engoliu algum valor
comparacao = pd.DataFrame({
    "antes": vi_df.isna().sum(),
    "depois": vi[vi_df.columns].isna().sum(),
})
comparacao["criados"] = comparacao["depois"] - comparacao["antes"]
comparacao

,antes,depois,criados
id,0,0,0
ip,0,0,0
ps,0,0,0
ts,159,159,0
des,0,0,0
date,0,0,0
energy,0,0,0
method,0,0,0
fullname,0,0,0
sigma_vi,157,157,0


In [25]:
ts_nulo = vi["ts"].isna()

# agrupa por objeto e método e conta quantos registros nulos cada um tem
vi.loc[ts_nulo].groupby(["des", "method"]).size()

des       method
101955    MC        157
2026 VY1  IOBS        1
29075     IOBS        1
dtype: int64

In [26]:
# duplicados: o mesmo objeto, na mesma data e método, não deve repetir
print("duplicados:", vi.duplicated(subset=["des", "date", "method"]).sum())

# probabilidade entre 0 e 1
print("ip fora de (0, 1]:", (~vi["ip"].between(0, 1, inclusive="right")).sum())

# integridade com o resumo: todo des do V deve existir no resumo, e vice-versa
des_vi = set(vi["des"])
des_resumo = set(resumo["des"])
print("des no V que não estão no resumo:", len(des_vi - des_resumo))
print("des do resumo sem nenhum registro no V:", len(des_resumo - des_vi))

# a quantidade de registros por objeto no V bate com n_imp do resumo?
qtd_v = vi.groupby("des").size().rename("qtd_v")
comparacao_n = resumo.set_index("des")[["n_imp"]].join(qtd_v)
print("objetos em que n_imp != qtd no V:", (comparacao_n["n_imp"] != comparacao_n["qtd_v"]).sum())

duplicados: 1752
ip fora de (0, 1]: 0
des no V que não estão no resumo: 0
des do resumo sem nenhum registro no V: 0
objetos em que n_imp != qtd no V: 0


In [27]:
# linhas 100% iguais em todas as colunas originais
print("linhas totalmente idênticas:", vi_df.duplicated().sum())

# as repetidas por (des, date, method), lado a lado
chave = ["des", "date", "method"]
repetidas = vi[vi.duplicated(subset=chave, keep=False)].sort_values(chave)
print("registros envolvidos:", len(repetidas), "| objetos:", repetidas["des"].nunique())
repetidas.head(10)

linhas totalmente idênticas: 0
registros envolvidos: 3232 | objetos: 482


,id,ip,ps,ts,des,date,energy,method,fullname,sigma_vi,sigma_mc,data_impacto
15872,a0101955,3.000000e-07,-4.68,<NA>,101955,2187-09-24.97,1422.0,MC,101955 Bennu (1999 RQ36),NaN,2.2858,2187-09-24
15873,a0101955,7.000000e-07,-4.32,<NA>,101955,2187-09-24.97,1421.0,MC,101955 Bennu (1999 RQ36),NaN,1.2905,2187-09-24
15870,a0101955,2.000000e-07,-4.86,<NA>,101955,2187-09-24.99,1422.0,MC,101955 Bennu (1999 RQ36),NaN,1.8608,2187-09-24
15871,a0101955,1.390000e-05,-3.02,<NA>,101955,2187-09-24.99,1422.0,MC,101955 Bennu (1999 RQ36),NaN,1.0423,2187-09-24
15856,a0101955,2.000000e-07,-4.87,<NA>,101955,2190-09-24.70,1422.0,MC,101955 Bennu (1999 RQ36),NaN,2.6052,2190-09-24
15857,a0101955,1.000000e-07,-5.17,<NA>,101955,2190-09-24.70,1422.0,MC,101955 Bennu (1999 RQ36),NaN,1.8779,2190-09-24
15845,a0101955,3.200000e-06,-3.67,<NA>,101955,2192-09-24.18,1422.0,MC,101955 Bennu (1999 RQ36),NaN,0.9203,2192-09-24
15846,a0101955,8.000000e-07,-4.27,<NA>,101955,2192-09-24.18,1422.0,MC,101955 Bennu (1999 RQ36),NaN,0.8623,2192-09-24
15840,a0101955,1.000000e-07,-5.18,<NA>,101955,2193-09-24.48,1421.0,MC,101955 Bennu (1999 RQ36),NaN,2.5056,2193-09-24
15841,a0101955,5.000000e-07,-4.48,<NA>,101955,2193-09-24.48,1421.0,MC,101955 Bennu (1999 RQ36),NaN,1.2362,2193-09-24


In [28]:
candidatas = {
    "des+date+method": ["des", "date", "method"],
    "+ ip": ["des", "date", "method", "ip"],
    "+ sigma_vi + sigma_mc": ["des", "date", "method", "sigma_vi", "sigma_mc"],
}
for nome, cols in candidatas.items():
    print(nome, "-> duplicados:", vi.duplicated(subset=cols).sum())

des+date+method -> duplicados: 1752
+ ip -> duplicados: 5
+ sigma_vi + sigma_mc -> duplicados: 2


In [29]:
print(resumo.loc[resumo["des"].isin(["2026 VY1", "29075"]), ["des", "n_imp", "ts_max"]])
vi.loc[vi["des"] == "2026 VY1", ["des", "date", "ip", "ps", "ts", "method"]]

           des  n_imp  ts_max
1685     29075      1    <NA>
2231  2026 VY1      2       0


,des,date,ip,ps,ts,method
2904,2026 VY1,2126-11-19.38,0.000007,-6.04,<NA>,IOBS
2905,2026 VY1,2122-11-19.30,0.000002,-6.69,0,IOBS


In [30]:
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT id, query_params, payload, ingested_at
        FROM bronze.sentry_raw
        WHERE source = 'sentry_removidos'
        ORDER BY id DESC
        LIMIT 1
        """
    )
    linha_id, params_r, payload_r, ingested_at_r = cur.fetchone()

print("linha:", linha_id, "| parâmetros:", params_r)
print("assinatura:", payload_r["signature"])
print("count:", payload_r["count"], "| registros em data:", len(payload_r["data"]))

# quais conjuntos de chaves existem
print(Counter(tuple(sorted(r.keys())) for r in payload_r["data"]))

rem_df = pd.DataFrame(payload_r["data"])
print(rem_df.shape)
display(rem_df.head())
rem_df.dtypes

linha: 17 | parâmetros: {'removed': 'true'}
assinatura: {'source': 'NASA/JPL Sentry Data API', 'version': '2.0'}
count: 3600 | registros em data: 3600
Counter({('des', 'removed'): 3600})
(3600, 2)


,des,removed
0,2026 VY15,2026-11-16 15:32:08
1,2026 VC6,2026-11-16 15:28:03
2,2026 VA1,2026-11-16 14:48:24
3,2026 OA3,2026-11-11 15:44:45
4,2026 VC3,2026-11-08 14:24:05


des        str
removed    str
dtype: object

In [32]:
rem = rem_df.copy()

# "2015-12-14 16:06" vira datetime
rem["removed"] = pd.to_datetime(rem["removed"], format="%Y-%m-%d %H:%M", errors="coerce")

# a conversão criou nulos?
print("nulos antes:", rem_df.isna().sum().to_dict())
print("nulos depois:", rem.isna().sum().to_dict())

# duplicados
print("des duplicados:", rem["des"].duplicated().sum())
print("des + removed duplicados:", rem.duplicated(subset=["des", "removed"]).sum())

# período coberto
print("removed:", rem["removed"].min(), "->", rem["removed"].max())

# algum removido ainda aparece no resumo atual?
print("removidos que também estão no resumo:", len(set(rem["des"]) & set(resumo["des"])))

nulos antes: {'des': 0, 'removed': 0}
nulos depois: {'des': 0, 'removed': 3600}
des duplicados: 0
des + removed duplicados: 0
removed: NaT -> NaT
removidos que também estão no resumo: 0


In [33]:
rem = rem_df.copy()

# agora com segundos, como o dado realmente vem
rem["removed"] = pd.to_datetime(rem["removed"], format="%Y-%m-%d %H:%M:%S")

print("nulos antes:", rem_df.isna().sum().to_dict())
print("nulos depois:", rem.isna().sum().to_dict())

print("removed:", rem["removed"].min(), "->", rem["removed"].max())
rem.dtypes

nulos antes: {'des': 0, 'removed': 0}
nulos depois: {'des': 0, 'removed': 0}
removed: 2002-01-15 22:51:46 -> 2026-11-16 15:32:08


des                   str
removed    datetime64[us]
dtype: object

In [34]:
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT query_params->>'des', payload
        FROM bronze.sentry_raw
        WHERE source = 'sentry_objeto'
        """
    )
    objetos = cur.fetchall()
print("objetos:", len(objetos))

# quais conjuntos de chaves existem no summary
for chaves, qtd in Counter(tuple(sorted(p["summary"].keys())) for _, p in objetos).items():
    print(qtd, "objetos com", len(chaves), "chaves:", chaves)

# o data do modo O repete os cenários do modo V? (qtd_v veio da célula 23)
qtd_o = pd.Series({des: len(p["data"]) for des, p in objetos}, name="qtd_o")
cmp = pd.concat([qtd_o, qtd_v], axis=1)
print("objetos em que O.data != V:", (cmp["qtd_o"] != cmp["qtd_v"]).sum())

objetos: 2251
2251 objetos com 23 chaves: ('cdate', 'darc', 'des', 'diameter', 'energy', 'first_obs', 'fullname', 'h', 'ip', 'last_obs', 'mass', 'method', 'n_imp', 'ndel', 'ndop', 'nobs', 'nsat', 'pdate', 'ps_cum', 'ps_max', 'ts_max', 'v_imp', 'v_inf')
objetos em que O.data != V: 0


In [36]:
exemplo = objetos[0][1]["summary"]
for campo, valor in exemplo.items():
    tipos = Counter(type(p["summary"][campo]).__name__ for _, p in objetos)
    print(f"{campo}: {valor!r}  | tipos: {dict(tipos)}")

h: '26.32'  | tipos: {'str': 2251}
ip: '2.937e-06'  | tipos: {'str': 2251}
des: '2011 TO'  | tipos: {'str': 2251}
darc: '469.18 days'  | tipos: {'str': 2251}
mass: '8.45e+06'  | tipos: {'str': 2251}
ndel: 0  | tipos: {'int': 2251}
ndop: 0  | tipos: {'int': 2251}
nobs: 44  | tipos: {'int': 2251}
nsat: '0'  | tipos: {'str': 2251}
cdate: '2022-09-14 20:48:30'  | tipos: {'str': 2251}
n_imp: 1  | tipos: {'int': 2251}
pdate: '2026-11-16 19:08:50'  | tipos: {'str': 2251}
v_imp: '14.04'  | tipos: {'str': 2251}
v_inf: '8.55'  | tipos: {'str': 2251}
energy: '1.983e-01'  | tipos: {'str': 2251}
method: 'IOBS'  | tipos: {'str': 2251}
ps_cum: '-6.15'  | tipos: {'str': 2251}
ps_max: '-6.15'  | tipos: {'str': 2251}
ts_max: '0'  | tipos: {'str': 2249, 'NoneType': 2}
diameter: '1.8e-02'  | tipos: {'str': 2251}
fullname: '(2011 TO)'  | tipos: {'str': 2251}
last_obs: '2013-01-12'  | tipos: {'str': 2251}
first_obs: '2011-10-01'  | tipos: {'str': 2251}
